# LFW Face Classification using MobileNetV2

Academic image-classification project. This notebook classifies LFW images among selected identities. It is not a webcam face-lock or face-verification application.

**Evaluation note:** train/validation/test are stratified by image, so identities occur in each split. This evaluates classification of known identities; it does not measure generalization to unseen identities.

In [ ]:
# Academic LFW Face Classification using MobileNetV2
# Run in Google Colab. Upload kaggle.json when prompted.

!pip -q install kaggle scikit-learn

from google.colab import files
import os, cv2, json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Kaggle credentials and dataset
uploaded = files.upload()
if "kaggle.json" not in uploaded:
    raise FileNotFoundError("Upload your Kaggle API token file named kaggle.json.")
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "wb") as f:
    f.write(uploaded["kaggle.json"])
os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d jessicali9530/lfw-dataset -p /content --force
!unzip -oq /content/lfw-dataset.zip -d /content

DATA_DIR = "/content/lfw-deepfunneled/lfw-deepfunneled"
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"LFW dataset folder not found: {DATA_DIR}")

# Keep identities with at least 20 images; stable class mapping.
MIN_IMAGES_PER_IDENTITY = 20
IMAGE_SIZE = (224, 224)
SEED = 42
tf.keras.utils.set_random_seed(SEED)

faces, labels, label_map = [], [], {}
for person in sorted(os.listdir(DATA_DIR)):
    person_dir = os.path.join(DATA_DIR, person)
    if not os.path.isdir(person_dir):
        continue
    image_names = sorted(os.listdir(person_dir))
    valid_names = [n for n in image_names if cv2.imread(os.path.join(person_dir, n)) is not None]
    if len(valid_names) < MIN_IMAGES_PER_IDENTITY:
        continue
    class_id = len(label_map)
    label_map[class_id] = person
    for image_name in valid_names:
        image = cv2.imread(os.path.join(person_dir, image_name))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, IMAGE_SIZE)
        faces.append(image)
        labels.append(class_id)

if len(label_map) < 2:
    raise ValueError("Need at least two identities with enough valid images.")

X = np.asarray(faces, dtype=np.float32)
y = np.asarray(labels, dtype=np.int32)
print("Images:", X.shape, "| identities:", len(label_map))

# Stratified image-level split: each retained identity appears in train/validation/test.
# This measures classification of known identities, not recognition of unseen people.
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.1765, random_state=SEED, stratify=y_trainval
)  # approximately 70% train / 15% validation / 15% test

# MobileNetV2 expects RGB values preprocessed to [-1, 1].
X_train = preprocess_input(X_train)
X_val = preprocess_input(X_val)
X_test = preprocess_input(X_test)

base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation="relu"),
    Dropout(0.4),
    Dense(len(label_map), activation="softmax")
])
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

# Augmentation only on training images; validation/test remain untouched.
train_datagen = ImageDataGenerator(
    rotation_range=15, zoom_range=0.15, horizontal_flip=True
)
early_stop = EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)
history = model.fit(
    train_datagen.flow(X_train, y_train, batch_size=32, shuffle=True, seed=SEED),
    epochs=15,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"Holdout test accuracy: {test_accuracy * 100:.2f}%")
print("Note: image-level split; test images belong to identities also present in training.")

os.makedirs("models", exist_ok=True)
os.makedirs("logs", exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = f"models/lfw_mobilenetv2_{timestamp}.keras"
model.save(model_path)
with open(f"logs/training_{timestamp}.json", "w") as f:
    json.dump({
        "test_accuracy": float(test_accuracy),
        "test_loss": float(test_loss),
        "epochs_run": len(history.history["loss"]),
        "num_classes": len(label_map),
        "seed": SEED,
        "class_map": label_map
    }, f, indent=2)
print("Saved model:", model_path)

plt.plot(history.history["accuracy"], label="Training")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.show()
